In [ ]:
!git config --global user.email estp.pl@gmail.com
!git config --global user.name LazyDart

In [ ]:
!git clone https://{token}@github.com/LazyDart/poleval-2024-qa.git

Cloning into 'poleval-2024-qa'...
remote: Enumerating objects: 194, done.
remote: Counting objects: 100% (194/194), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 194 (delta 87), reused 163 (delta 60), pack-reused 0
Receiving objects: 100% (194/194), 657.13 KiB | 2.02 MiB/s, done.
Resolving deltas: 100% (87/87), done.


In [ ]:
!python ./poleval-2024-qa/scripts/t5/load_t5.py --kind large --colab

tokenizer_config.json: 100% 141/141 [00:00<00:00, 692kB/s]
spiece.model: 100% 1.12M/1.12M [00:00<00:00, 21.0MB/s]
special_tokens_map.json: 100% 65.0/65.0 [00:00<00:00, 411kB/s]
/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
config.json: 100% 660/660 [00:00<00:00, 3.88MB/s]
pytorch_model.bin: 100% 3.28G/3.28G [00:27<00:00, 120MB/s]


In [ ]:
!pip install wandb datasets

In [ ]:
!pip install accelerate -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.1/314.1 kB 6.1 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import wandb
import pandas as pd
from transformers import Trainer, Seq2SeqTrainer, Seq2SeqTrainingArguments

import os
from datetime import datetime

os.chdir("./poleval-2024-qa/scripts")

from data_processing import poquad, processing
from t5.load_t5 import *

In [ ]:
!ls

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
poquad.save_poquad_manually_downloaded("../data/poquad-downloaded-manually/poquad-train.json", "../data/poquad-downloaded-manually/poquad-dev.json")

In [ ]:
train_df, valid_df = poquad.load_poquad_manually_downloaded("../data/poquad-manually-processed/")

In [ ]:
train_input = poquad.dataset_into_str_input(train_df)

In [ ]:
valid_input = poquad.dataset_into_str_input(valid_df)

In [ ]:
tokenizer, model = load_plt5("../models/plt5-original-large")

In [ ]:
from transformers import TrainerCallback

class PrintMemoryUsageCallback(TrainerCallback):
    """ Callback that prints memory allocation during training """
    def on_step_end(self, args, state, control, **kwargs):
        print(f"Step {state.global_step}: {torch.cuda.memory_allocated() / 1024 ** 2:.2f} MB allocated")

In [ ]:
wandb.init(
    # set the wandb project where this run will be logged
    project="PLT5 Large Finetuning Poquad",
    # track hyperparameters and run metadata
    # config={
    # "architecture": "PLT5 Small",
    # "dataset": "Poquad",
    # }
)

wandb: Currently logged in as: m-tkacz6 (m-tkacz-uw). Use `wandb login --relogin` to force relogin


In [ ]:
torch.cuda.empty_cache()

In [ ]:
train_dataset = processing.TextDataset(train_input, tokenizer, 1024, 128)
valid_dataset = processing.TextDataset(valid_input, tokenizer, 1024, 128)

num_train_samples = train_dataset.__len__()
train_batch_size = 4
gradient_accumulation_steps = 1

num_train_steps_per_epoch = (num_train_samples // train_batch_size // gradient_accumulation_steps) + 1

# Calculate save steps for every 2 epochs
save_steps = 2 * num_train_steps_per_epoch

# Initialize model

# Define TrainingArguments
training_args = Seq2SeqTrainingArguments(
    learning_rate=3e-4,
    output_dir='./results',
    resume_from_checkpoint="../models/plt5-large-2epochs",
    run_name=f'plt5-large-{datetime.now().strftime("%Y-%m-%d_%H-%M-%S")}',
    num_train_epochs=4,
    per_device_train_batch_size=4,
    # per_device_eval_batch_size=4,
    save_total_limit=4,
    warmup_steps=500,
    weight_decay=0.01,
    gradient_accumulation_steps=gradient_accumulation_steps,  # Gradient accumulation steps
    logging_dir='./logs',
    overwrite_output_dir=True,
    bf16=True,
    save_steps=save_steps,  # Save checkpoint every 2 epochs
    # metric_for_best_model="eval_loss",  # Use evaluation loss to determine the best model
    # greater_is_better=False,  # Lower eval_loss is better
    logging_steps=100,  # Log every 100 steps
    predict_with_generate=True,  # Use generate method for predictions
)


# Initialize Trainer
trainer = Seq2SeqTrainer(
        model=model,
        tokenizer=tokenizer,
        args=training_args,
        train_dataset=train_dataset,
        # eval_dataset=valid_dataset,
        # data_collator=data_collator,
        callbacks=[PrintMemoryUsageCallback()],
        )

# # Train the model
trainer.train()

# Save the model
trainer.save_model('./poleval-2024-qa/models/plt5-large-8epochs')

# print("Training complete and model saved")
wandb.finish()

Step 41287: 9596.40 MB allocated


Step,Training Loss
100,80.349500
200,50.127000
300,33.886600
400,13.675900
500,1.800900
600,0.568000
700,0.545400
800,0.483100
900,0.460100
1000,0.242900


Streaming output truncated to the last 5000 lines.
Step 44025: 9596.40 MB allocated
Step 44026: 9596.40 MB allocated
Step 44027: 9596.40 MB allocated
Step 44028: 9596.40 MB allocated
Step 44029: 9596.40 MB allocated
Step 44030: 9596.40 MB allocated
Step 44031: 9596.40 MB allocated
Step 44032: 9596.40 MB allocated
Step 44033: 9596.40 MB allocated
Step 44034: 9596.40 MB allocated
Step 44035: 9596.40 MB allocated
Step 44036: 9596.40 MB allocated
Step 44037: 9596.40 MB allocated
Step 44038: 9596.40 MB allocated
Step 44039: 9596.40 MB allocated
Step 44040: 9596.40 MB allocated
Step 44041: 9596.40 MB allocated
Step 44042: 9596.40 MB allocated
Step 44043: 9596.40 MB allocated
Step 44044: 9596.40 MB allocated
Step 44045: 9596.40 MB allocated
Step 44046: 9596.40 MB allocated
Step 44047: 9596.40 MB allocated
Step 44048: 9596.40 MB allocated
Step 44049: 9596.40 MB allocated
Step 44050: 9596.40 MB allocated
Step 44051: 9596.40 MB allocated
Step 44052: 9596.40 MB allocated
Step 44053: 9596.40 MB al

In [ ]:
from google.colab import files
from pathlib import Path

pth = Path("./poleval-2024-qa/models/plt5-large-8epochs")

for file in pth.iterdir():
  files.download(file)